# Plotting Gradients on Brain Surface (WIP)
By default, this script uses the Schaefer aligned template connectome from the Human Connectome Project, available through the `Branspace` python package - see [here]( https://brainspace.readthedocs.io/en/latest/generated/brainspace.datasets.load_group_fc.html#brainspace.datasets.load_group_fc ) for more details. 

In [ ]:

# Enter paths to your subject text file and lookup table below
lut_path = "regions_100_fixed.tsv"
subjects = "adni_subs.txt"
results = Path("results")
num_regions = 100
template_connectome_path = ""
################ Default Settings, edit if you want ##############
template_gradient_map_kwargs = {
    "n_components": 10,
    "approach": "dm",
    "random_state": 0,
    "kernel": "cosine",
}
gradient_map_kwargs = {
    "n_components": 10,
    "approach": "dm",
    "random_state": 0,
    "kernel": "cosine",
    "alignment": "procrustes",
}
n_iters = 100
sparsity = None
lut = pd.read_csv(lut_path, index_col=0, delimiter="\t")
numbers_array = np.arange(1, num_regions + 1)
include_networks = None  # Change to a list of networks to exclude from plots

if not template_connectome_path:
    template_connectome = load_group_fc("schaefer", num_regions)
else:
    # Gradient generation below
    template_connectome = pd.read_csv(
        template_connectome_path, index_col=0
    ).to_numpy()
if template_connectome.shape[0] != template_connectome.shape[1]:
    raise ValueError(
        "Ensure the first column of the template contains the region labels"
    )

template_conn_fish = get_fisher(template_connectome)
template_gm = GradientMaps(**template_gradient_map_kwargs)
template_gm.fit(template_conn_fish, sparsity=0.8)

results.mkdir(parents=True, exist_ok=True)
plot_connectome(template_connectome, lut, results)
with open(subjects, "r") as f:
    sub_list = f.readlines()
subs = [Path(x.removesuffix("\n")) for x in sub_list]

all_conn = []
for path in subs:
    conn = pd.read_csv(path, index_col=0).to_numpy()
    if conn.shape[0] != conn.shape[1]:
        raise ValueError(
            "Ensure data is saved with the first column being region name"
        )

    conn = get_fisher(conn)
    thresholds = np.percentile(conn, 80, axis=1, keepdims=True)
    mask = conn <= thresholds
    conn[mask] = 0
    all_conn.append(conn)
fish_conn = np.array(all_conn).mean(axis=0)
gm = GradientMaps(**gradient_map_kwargs)
gm.fit(
    fish_conn,
    reference=template_gm.gradients_,
    sparsity=sparsity,
    n_iter=n_iters,
)

save_path = results.joinpath(path.name.split(".")[0])
save_path.mkdir(parents=True, exist_ok=True)
plot_connectome(conn, lut, save_path)
plot_gradient_features(gm, save_path)
plot_gradient_space(gm.aligned_, lut, include_networks, save_path)  # type:ignore
np.savetxt(
    save_path.joinpath("gradients.csv"),
    gm.aligned_,  # type:ignore
    delimiter=",",
)

labeling = load_parcellation("schaefer", 100, join=True)

surf_lh, surf_rh = load_conte69()
to_plot = []
for k in range(3):
    to_plot.append(
        map_to_labels(
            gm.aligned_[:, k],
            labeling,
            mask=labeling != 0,
            fill=np.nan,
        )
    )

plot_hemispheres(
    surf_lh,
    surf_rh,
    array_name=to_plot,
    size=(1250, 400),
    cmap="viridis_r",
    color_bar=True,
    label_text=["Grad1", "Grad2", "Grad3"],
    zoom=1.55,
    screenshot=False,
    interactive=False,
    transparent_bg=False,
)